In [ ]:
import time

In [ ]:
import importlib
from src import preprocess, encoded_tokens, models
importlib.reload(preprocess)
importlib.reload(encoded_tokens)
importlib.reload(models)

import pandas as pd
from src.preprocess import load_and_preprocess
from src.encoded_tokens import build_vocab, text_to_sequence, pad_sequence
from src.models import LSTMModel, GRUModel, BIGRUModel, CNNModel, CNN_GRU_Model


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset , DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [ ]:
#read the preprocessed data
texts , labels = load_and_preprocess(
    df_path = "/content/drive/MyDrive/Sentiment_classification/data/IMDB Dataset.csv",
    text_column = "review",
    label_column = "sentiment",
    mode = "embedding"
)

In [ ]:
texts.head()

In [ ]:
labels


In [ ]:
#build vocabulary
vocab = build_vocab(texts,max_vocab_size = 15_000)
#convert text to encoded sequence
sequences = text_to_sequence(texts, vocab)
#Truncate extra length sequence and pad the shorter sequences
padded_seq = pad_sequence(sequences, seq_len= 250)



In [ ]:
#convert padded seqences to tensors
X = torch.tensor(padded_seq, dtype = torch.long)
y = torch.tensor(labels, dtype = torch.float32)

In [ ]:
#split the data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [ ]:
#create TensorDataset and DataLoaders
train_set = TensorDataset(X_train, y_train)
test_set = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 64)


In [ ]:
#Model object for Sentiment classification
Sentimentcnn_gru = CNN_GRU_Model(len(vocab), embedding_size = 100)
model = Sentimentcnn_gru
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())


In [ ]:
#Train
start = time.time()
epochs = 15
for epoch in range(epochs):
  model.train()
  running_loss = 0.0
  for xb, yb in train_loader:
    #set the gradient to zero
    optimizer.zero_grad()
    #pass input to the model
    output = model(xb)
    #Apply sigmoid on squeezed output
    output = torch.sigmoid(output.squeeze())
    #Calculate the loss
    loss = criterion(output, yb)
    #Backward propagation
    loss.backward()
    #update the gradients
    optimizer.step()
    #update the loss
    running_loss+= loss.item()

  print(f"epoch {epoch+1}/{epochs} and loss {running_loss/len(train_loader)} ")
end = time.time()

In [ ]:
correct_values = 0
total_values = 0
with torch.no_grad():
  model.eval()
  for xb, yb in test_loader:
    #pass input to the model
    output = model(xb)
    #Predicted Sentiment
    predicted = (torch.sigmoid(output.squeeze())>0.5).float()
    #update the correct predicted values
    correct_values += (predicted == yb).sum().item()
    #update the count of values in each batch
    total_values += yb.size(0)
  accuracy = correct_values/total_values*100
  print(f"Accuracy: {accuracy}")


In [ ]:
results = {
    'Model' : "CNN_GRU (T4 GPU)",
    'vocab_size': len(vocab),
    'embedding_size': 100,
    'sequence_length': 250,
    'hidden_size': '128, 100_filters,',
    'num_layers' :1,
    'batch_size': 64,
    'num_epochs': 15,
    'Accuracy' :accuracy,
    'Time' : end-start
}

In [ ]:
new_df = pd.DataFrame([results])

In [ ]:
new_df


In [ ]:
file_path = "/content/drive/MyDrive/Sentiment_classification/results/embedding_results.csv"
results_df = pd.read_csv(file_path)

In [ ]:
results_df = pd.concat([results_df, new_df], ignore_index = True)

In [ ]:
results_df.to_csv(file_path, index = False)